# นับดอกไม้ + บอกชนิด จากหลายรูป (v5 = YOLO + BioCLIP 2 เท่านั้น, เน้นดอกไม้จริง)

> **เวอร์ชัน v5.1** (2026-10-28) — เวอร์ชันเก่าอยู่ในโฟลเดอร์ `version/`

```
เพิ่มรูป (หลายรูป/หลายรอบ) → กด "ตรวจทั้งหมด"
   แต่ละรูป: เป็นดอกไม้ไหม? (BioCLIP 2) → ไม่ใช่ = แจ้งเตือน
             ใช่ → YOLO หาทีละดอก → crop → ตัดของที่ไม่ใช่ดอกไม้ (BioCLIP 2)
                 → BioCLIP 2 ทายชนิดทีละดอก (ชื่อวิทยาศาสตร์ + รูปตัวอย่าง + บริบททั้งรูป)
   สุดท้าย: สรุปรวมทุกรูป (ดอกทั้งหมดกี่ดอก ชนิดละกี่ดอก อยู่ในกี่รูป)
```

**สิ่งที่เปลี่ยนจากเวอร์ชันเดิม**
1. prompt ของ BioCLIP ใช้ **ชื่อวิทยาศาสตร์** (เช่น *Tulipa gesneriana*) เพราะ BioCLIP เทรนด้วยชื่อแบบนี้
2. อัปเกรดเป็น **BioCLIP 2** (ทายชนิดแม่นกว่ารุ่นแรกมาก)
3. ผสม **บริบทของทั้งรูป** เข้ากับแต่ละดอก (`CONTEXT_WEIGHT`)
4. ใส่ **รูปตัวอย่าง** ของแต่ละชนิดได้ (`refs/<ชนิด>/*.jpg`) แม่นที่สุดในบรรดาทุกข้อ
5. Detector เริ่มต้นเป็น **OWLv2** (zero-shot หาดอกไม้ได้ทุกแบบ) หรือ YOLO ที่ train แบบ `single_cls`
6. ตัดกล่องที่ครอบทั้งช่อออก ไม่ให้นับเป็น 1 ดอก
7. **(v4)** ปฏิเสธดอกไม้การ์ตูน/ในเกม และดอกไม้ปลอมให้ใช้ CLIP ViT-L-14 ช่วยบอกชนิด (โหลดเฉพาะตอนเจอดอกไม้ปลอม)
8. **(v4.1) นับทีละดอกแม่นขึ้น** ทดสอบกับ 15 รูป ผิดรวมจาก ~65 ดอก เหลือ ~10 ดอก
   - OWLv2 ค้นด้วย `"flower"` + ชื่อดอกไม้ทุกชนิดใน `flowers` (เดิม `"a photo of a flower"` ได้กล่องครอบทั้งรูป แล้วกล่องดอกจริงถูกตัดทิ้ง)
   - เกณฑ์คะแนนปรับตามรูป (`DET_REL` × คะแนนสูงสุดในรูป) เพราะคะแนน OWLv2 ต่างกันมากในแต่ละรูป
   - กล่องซ้อนกัน (กลีบในดอก / ดอก+ก้าน) เหลือกล่องเดียว และเลือกกล่องที่เป็นตัวดอก
   - ภาพถ่ายจริงพื้นดำไม่ถูกปัดเป็นการ์ตูนแล้ว และดอกจริงไม่ถูกเรียกว่าดอกไม้ปลอม (`DRAWN_THR`, `ARTIFICIAL_THR` = 0.8)
9. **(v4.2) YOLO นับแม่นขึ้น**
   - `YOLO_CONF` 0.25 → 0.15 และเปิด TTA (`YOLO_TTA`) หาดอกที่เคยหลุด (ดอกด้านหลัง/ชนิดที่ไม่มีใน dataset)
   - ตัดกล่องเล็กผิดปกติ (`MIN_REL_AREA` × ขนาดกลางของกล่องในรูป) = เศษกลีบ ไม่ใช่ดอก
   - ด่านกรอง crop รู้จักริบบิ้น/กระดาษห่อ ไม่นับเป็นดอก
10. **(v4.3) รันใน VS Code ได้** อ่านรูปจากโฟลเดอร์ (`IMAGE_DIR`) แทนการอัปโหลด, รูปตัวอย่างวางใน `refs/<ชนิด>/`
11. **(v4.4) เลือก detector ได้ 3 แบบ** `"yolo"`, `"owl2"`, `"yolo+owl2"` (YOLO เป็นหลัก + เติมดอกที่ OWLv2 เจอแต่ YOLO ไม่เจอ)
   - แก้บั๊ก: ดอกริมรูปที่มีเศษกลีบซ้อนอยู่ถูกมองเป็นกล่องทั้งช่อแล้วถูกตัดทิ้ง (ตัดเศษกลีบก่อนแล้ว)
12. **(v4.5) ช่อดอกไม้ผสมหลายชนิด + เซลล์สรุปไม่ค้าง**
   - ใช้บริบททั้งรูปเฉพาะรูปที่เป็นดอกชนิดเดียวชัดๆ (`CONTEXT_MIN`) ช่อผสมเคยถูกดึงไปเป็นชนิดเดียวกันหมด/กลายเป็น unknown
   - `CROP_FLOWER_THR` 0.25 → 0.5 ตัดกล่องแจกัน/ใบไม้/กิ่ง (ได้ 0.03–0.35 ส่วนดอกจริงได้ ≥ 0.54)
   - โหลด CLIP ของดอกไม้ปลอมไว้ตั้งแต่เซลล์โหลดโมเดล (เดิมไปดาวน์โหลด 1.7GB กลางเซลล์สรุป ดูเหมือนค้าง) + บอกความคืบหน้าทีละรูป
13. **(v4.6)** `yolo+owl2`: ดอกใหญ่ที่ OWLv2 เจอแต่มีกล่องเล็กของ YOLO (เช่น ยิปโซ) อยู่ข้างใน ไม่ถูกตัดทิ้งแล้ว (ทานตะวันในช่อผสมเคยหาย)
   - เพิ่มชนิด goldenrod, scabiosa, clematis, craspedia (เจอบ่อยในช่อผสม เดิมขึ้น unknown)
14. **(v4.7)** `yolo+owl2`: กล่อง OWLv2 ที่ไม่มั่นใจ (เช่น กล่องแจกันที่มีทิวลิปอยู่ด้านบน) ไม่แทนกล่อง YOLO แล้ว
15. **(v4.8) ดอกช่อ** (ดอกเข็ม ไฮเดรนเยีย snapdragon ฯลฯ) นับ 1 ช่อ = 1 ดอกได้ (`COUNT_CLUSTER_AS_ONE`) เดิมกล่องทั้งช่อถูกตัดทิ้ง เหลือแต่ดอกย่อยกระจัดกระจาย
   - **บอกสีของแต่ละดอก** (RGB + ชื่อสี) ตัดพื้นหลังสีเขียว/มืดออกก่อน แล้วใช้ค่ากลางของพิกเซลที่เหลือ
   - เซลล์โหลดโมเดลดาวน์โหลด CLIP ดอกไม้ปลอมลงดิสก์อย่างเดียว ไม่โหลดเข้า RAM (ประหยัด ~1.7GB ตอนโหลด) และข้ามได้ถ้า `ARTIFICIAL_CLIP_WEIGHT = 0`
   - `CONTEXT_WEIGHT` 0.3 → 0.5 (ใช้เฉพาะรูปชนิดเดียวอยู่แล้ว) ดอกย่อยของดอกเข็มเคยถูกทายเป็นลีลาวดี
16. **(v4.9)** เปลี่ยน YOLO เป็น **YOLO11s** (`runs/detect/flower_single_yolo11/`) ใช้ dataset เดิม train แบบคลาสเดียวเหมือนเดิม
17. **(v5.0) เหลือ 2 โมเดล: YOLO (หา+นับ) + BioCLIP 2 (บอกชนิด + ด่านกรองว่าเป็นดอกไม้ไหม)** เน้นดอกไม้จริงให้แม่นก่อน
   - ตัด OWLv2, CLIP ViT-B-32 (ด่านกรอง), CLIP ViT-L-14 (ดอกไม้ปลอม) ออก
   - ตัดการแยกดอกจริง/ดอกปลอม/การ์ตูน ออก (ดอกไม้ปลอม/การ์ตูนส่วนใหญ่ไม่ผ่านด่านกรองของ BioCLIP เอง เช่น ทานตะวันลวดกำมะหยี่ได้ 0.19)
   - ด่านกรองใช้ BioCLIP 2 แทน CLIP และตัด prompt `"a photo of fruit"` (ดึงลีลาวดี/ทานตะวันจริงไปเป็นผลไม้)
     ทดสอบ 22 รูป: ดอกจริงทั้งรูป ≥ 0.67, กล่องดอกจริง ≥ 0.63, กล่องที่ไม่ใช่ดอก ≤ 0.22
18. **(v5.1)** ทดสอบ **BioCLIP 2.5 Huge** (ViT-H/14, ส.ค. 2026) แล้ว **ยังไม่ใช้**: ทายชนิดมั่นใจกว่า แต่ด่านกรองพัง
   (ปัดรูปดอกจริงตก 5/22 เช่น ทิวลิป 14 ดอกได้ 0.20 แต่ปล่อยดอกลวดกำมะหยี่ผ่าน 0.99) | ช้ากว่า ~1.5 เท่า
   - GPU ใช้ fp16 (ประหยัดแรมการ์ด) | แต่ละ crop เข้า BioCLIP รอบเดียว ใช้ทั้งด่านกรองและทายชนิด (เดิม 2 รอบ)

รันทีละเซลล์จากบนลงล่าง — **Colab**: Runtime → GPU | **VS Code**: เลือก kernel Python 3.10 แล้วรันได้เลย (รูปอ่านจาก `IMAGE_DIR`, รูปตัวอย่างวางใน `refs/<ชนิด>/`, YOLO ต้องมี `YOLO_WEIGHTS`)

## 1) ติดตั้ง

In [ ]:
%pip install -q open_clip_torch ultralytics

In [ ]:
# train YOLO (ข้ามอัตโนมัติถ้ามี best.pt แล้ว)
%pip install -q roboflow
import os, torch

# ตั้งค่า YOLO11 (รุ่นก่อน: "flower_single_yolo11" imgsz 640 / "flower_single" YOLOv8)
YOLO_RUN = "flower_single_yolo11_960"
YOLO_BASE = "yolo11s.pt"
if os.path.exists(f"runs/detect/{YOLO_RUN}/weights/best.pt"):
    print(f"มีโมเดลที่ train แล้ว ข้ามการ train (ลบโฟลเดอร์ runs/detect/{YOLO_RUN} ถ้าจะ train ใหม่)")
elif not torch.cuda.is_available():   # train บน CPU ใช้หลายชั่วโมง
    print(f"⚠️ ไม่มี GPU: ดาวน์โหลด best.pt จาก Colab มาวางที่ runs/detect/{YOLO_RUN}/weights/best.pt")
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])   # ตั้งครั้งเดียว: setx ROBOFLOW_API_KEY "<key>" (Colab: os.environ[...] = "<key>")
    project = rf.workspace("elbrusds-lzhr7").project("bouquet")
    dataset = project.version(8).download("yolov11")

    from ultralytics import YOLO
    model = YOLO(YOLO_BASE)   # s ใหญ่กว่า n นิดหน่อย แต่หาดอกได้ครบกว่า
    # single_cls=True → ให้ YOLO ทำหน้าที่ "หาว่าดอกไม้อยู่ตรงไหน" อย่างเดียว ส่วนการบอกชนิดให้ BioCLIP ทำ
    # data.yaml ของ Roboflow ใช้ path ../train ที่ ultralytics หาไม่เจอ → เขียน yaml ใหม่แบบ path เต็ม
    yaml = f"{dataset.location}/data_local.yaml"
    with open(yaml, "w") as f:
        f.write(f"path: {dataset.location}\ntrain: train/images\nval: valid/images\nnc: 1\nnames: ['flower']\n")
    # imgsz 960: ดอกเล็กในช่อหาเจอขึ้น | batch=-1: เลือก batch ตาม VRAM | patience: หยุดเองถ้าไม่ดีขึ้น 30 รอบ
    model.train(data=yaml, epochs=120, imgsz=960, batch=-1, patience=30,
                single_cls=True, name=YOLO_RUN, exist_ok=True)

## 2) ตั้งค่า

In [ ]:
# ---------- ตัวตรวจจับ (YOLO หา "ตำแหน่ง" ดอกไม้) ----------
YOLO_WEIGHTS = "../runs/detect/flower_single_yolo11/weights/best.pt"   # YOLOv8 เดิม: "runs/detect/flower_single/weights/best.pt"

YOLO_CONF = 0.15          # คะแนนขั้นต่ำของ YOLO (0.25 หลุดดอกด้านหลัง/ดอกที่ไม่มีใน dataset)
YOLO_TTA = True          # (YOLO) ทายหลายสเกล + กลับซ้ายขวา แล้วรวมผล เจอดอกครบขึ้น แต่ช้าลง ~2–3 เท่า
DET_IOU = 0.3
MIN_BOX_PX = 16
MIN_REL_AREA = 0.15      # ตัดกล่องที่เล็กกว่า 15% ของกล่องขนาดกลางในรูป (เศษกลีบ) | 0 = ปิด
DET_MAX_AREA = 0.9
DROP_GROUP_BOXES = True   # ตัดกล่องใหญ่ที่ครอบหลายดอก (ทั้งช่อ) ไม่ให้นับเป็น 1 ดอก
NEST_COVER = 0.7          # กล่องที่อยู่ในอีกกล่อง ≥ 70% = ดอกเดียวกัน (กลีบในดอก, ดอก+ก้าน) → นับ 1
INNER_KEEP = 0.8          # กล่องในได้คะแนน ≥ 80% ของกล่องนอก → เลือกกล่องใน (ตัวดอก ไม่เอาก้าน/ใบ)

# ---------- ตัวจำแนกชนิด (บอกว่าเป็น "ดอกอะไร") ----------
SPECIES_MODEL = "hf-hub:imageomics/bioclip-2"   # "hf-hub:imageomics/bioclip-2.5-vith14" ทายชนิดมั่นใจกว่า แต่ใช้เป็นด่านกรองไม่ได้ (ดู v5.1)
SPECIES_THR = 0.35     # ความมั่นใจต่ำกว่านี้ = unknown
CONTEXT_WEIGHT = 0.5   # ผสมผลของ "ทั้งรูป" เข้ากับแต่ละดอก (0 = ปิด) ใช้เฉพาะรูปชนิดเดียว (ดอกเข็มที่ 0.3 ถูกทายเป็นลีลาวดี)
CONTEXT_MIN = 0.8      # ใช้บริบทเฉพาะเมื่อทั้งรูปเป็นชนิดเดียวชัดๆ (รูปชนิดเดียวได้ ~1.0, ช่อผสมได้ 0.36–0.60)
REF_DIR = "../refs"       # รูปตัวอย่าง refs/<ชื่อชนิด>/*.jpg (ไม่มีก็ได้ ดูเซลล์ 5.1)
REF_WEIGHT = 0.6       # น้ำหนักของรูปตัวอย่าง เทียบกับข้อความ (0–1)
SHOW_TOP3 = True       # พิมพ์ 3 อันดับแรกของแต่ละดอก ไว้ดูว่าโมเดลลังเลระหว่างอะไร

# ---------- ด่านกรอง "เป็นดอกไม้ไหม" (BioCLIP 2) ----------
IMAGE_FLOWER_THR = 0.30   # ดอกจริงทั้งรูปได้ ≥ 0.67, ทานตะวันลวดกำมะหยี่ได้ 0.19
CROP_FLOWER_THR = 0.5     # กล่องที่ไม่ใช่ดอก (ใบ/ห่อ/ริบบิ้น) ได้ ≤ 0.22, ดอกจริงได้ ≥ 0.63
UNKNOWN = "unknown"

# ---------- ดอกช่อ / ดอกช่อยาว ----------
COUNT_CLUSTER_AS_ONE = True   # True = 1 ช่อ = 1 ดอก | False = นับทีละกล่องดอกย่อย
CLUSTER_SPECIES = {"ixora", "hydrangea", "lilac", "snapdragon", "delphinium", "lavender", "hyacinth",
                   "gladiolus", "stock", "baby's breath", "goldenrod", "statice", "lily of the valley", "bougainvillea"}

## 3) โหลดโมเดล

In [ ]:
import os
import torch, open_clip
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# BioCLIP 2: ทายชนิดดอกไม้ + ด่านกรองว่าเป็นดอกไม้ไหม (เทรนจากภาพสิ่งมีชีวิตจริงกว่า 200 ล้านรูป ด้วยชื่อวิทยาศาสตร์)
# GPU ใช้ fp16: ประหยัดแรมการ์ดครึ่งหนึ่ง (BioCLIP 2.5 แบบ fp32 ใส่การ์ด 4GB ไม่ได้)
bio_model, _, bio_preprocess = open_clip.create_model_and_transforms(
    SPECIES_MODEL, precision="fp16" if device == "cuda" else "fp32")
bio_tok = open_clip.get_tokenizer(SPECIES_MODEL)
bio_model = bio_model.to(device).eval()

# YOLO: หาตำแหน่งดอกไม้ (train แบบ single_cls ในเซลล์ 1)
if not os.path.exists(YOLO_WEIGHTS):
    raise FileNotFoundError(f"ไม่มีไฟล์ {YOLO_WEIGHTS} — train ในเซลล์ 1 (ต้องมี GPU เช่น Colab) แล้ววาง best.pt ไว้ที่นี่")
yolo_det = YOLO(YOLO_WEIGHTS)

print("Models loaded! detector =", YOLO_WEIGHTS, "| species model =", SPECIES_MODEL)


## 4) ฟังก์ชันช่วย: embedding / detect / crop

In [ ]:
@torch.inference_mode()
def encode_text(model, tok, prompts):
    t = tok(prompts).to(device)
    f = model.encode_text(t)
    return f / f.norm(dim=-1, keepdim=True)


@torch.inference_mode()
def encode_image(model, preprocess, img):
    # dtype จากชั้นแรกของ image encoder (โหมด fp16 บางพารามิเตอร์ยังเป็น fp32)
    x = preprocess(img).unsqueeze(0).to(device, model.visual.conv1.weight.dtype)
    f = model.encode_image(x)
    return (f / f.norm(dim=-1, keepdim=True))[0]


def _covered(inner, outer):
    """สัดส่วนพื้นที่ของกล่อง inner ที่อยู่ในกล่อง outer"""
    ix0, iy0 = max(inner[0], outer[0]), max(inner[1], outer[1])
    ix1, iy1 = min(inner[2], outer[2]), min(inner[3], outer[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    area = max(1e-6, (inner[2] - inner[0]) * (inner[3] - inner[1]))
    return inter / area


def drop_group_boxes(cand, min_inside=2):
    """ตัดกล่องที่มีกล่องอื่นอยู่ข้างในตั้งแต่ 2 กล่องขึ้นไป (มักเป็นกล่องทั้งช่อ ไม่ใช่ดอกเดียว)"""
    out = []
    for i, (bi, si) in enumerate(cand):
        inside = sum(1 for j, (bj, _) in enumerate(cand) if j != i and _covered(bj, bi) >= NEST_COVER)
        if inside < min_inside:
            out.append((bi, si))
    return out


def drop_nested_boxes(cand):
    """กล่องที่ซ้อนกันเกิน NEST_COVER = ดอกเดียวกัน → เหลือกล่องเดียว
    ปกติเก็บกล่องที่คะแนนสูงกว่า แต่ถ้ากล่องในคะแนนใกล้กัน (≥ INNER_KEEP) เลือกกล่องใน เพราะกล่องนอกมักรวมก้าน/ใบ"""
    out = []
    for bi, si in sorted(cand, key=lambda c: -c[1]):
        j = next((j for j, (bj, _) in enumerate(out)
                  if _covered(bi, bj) >= NEST_COVER or _covered(bj, bi) >= NEST_COVER), None)
        if j is None:
            out.append((bi, si))
        elif _covered(bi, out[j][0]) >= NEST_COVER and si >= INNER_KEEP * out[j][1]:
            out[j] = (bi, si)
    return out


def drop_tiny_boxes(cand):
    """ตัดกล่องที่เล็กผิดปกติเทียบกับกล่องขนาดกลางในรูป (มักเป็นเศษกลีบของดอกข้างๆ)
    ต้องมี ≥ 3 กล่อง ไม่งั้นค่ากลางไม่มีความหมาย"""
    rel = MIN_REL_AREA
    if rel <= 0 or len(cand) < 3:
        return cand
    areas = sorted((b[2] - b[0]) * (b[3] - b[1]) for b, _ in cand)
    med = areas[len(areas) // 2]
    return [(b, s) for b, s in cand if (b[2] - b[0]) * (b[3] - b[1]) >= rel * med]


def _yolo_cand(img):
    """YOLO → list ของ ((x0, y0, x1, y1), score)"""
    results = yolo_det(img, conf=YOLO_CONF, iou=DET_IOU, augment=YOLO_TTA, verbose=False)[0]
    return [(box.xyxy[0].tolist(), box.conf[0].item()) for box in results.boxes]


def detect_flowers(img):
    """ตรวจจับดอกไม้ด้วย YOLO → (กล่องดอก, กล่องที่ครอบหลายกล่อง) แต่ละอันเป็น list ของ ((x0, y0, x1, y1), score)"""
    W, H = img.size
    cand = _yolo_cand(img)

    # กรองกล่องที่เล็กหรือใหญ่เกินไป
    out = []
    for (x0, y0, x1, y1), score in cand:
        w, h = x1 - x0, y1 - y0
        if w < MIN_BOX_PX or h < MIN_BOX_PX:
            continue
        if w * h > DET_MAX_AREA * W * H:
            continue
        out.append(((x0, y0, x1, y1), score))

    out = drop_tiny_boxes(out)   # ก่อน drop_group_boxes: เศษกลีบในดอกริมรูปทำให้ดอกจริงถูกมองเป็นกล่องทั้งช่อ
    groups = []
    if DROP_GROUP_BOXES:
        kept = drop_group_boxes(out)
        groups = [c for c in out if c not in kept]   # เก็บไว้ให้ merge_clusters (ดอกช่อ เช่น ดอกเข็ม คือกล่องแบบนี้)
        out = kept
    return drop_nested_boxes(out), groups


def crop_with_padding(img, box, pad=0.1):
    x0, y0, x1, y1 = box
    W, H = img.size
    px, py = (x1 - x0) * pad, (y1 - y0) * pad
    return img.crop((max(0, int(x0 - px)), max(0, int(y0 - py)),
                     min(W, int(x1 + px)), min(H, int(y1 + py))))

## 5) รายชื่อดอกไม้ + prompt
`flowers` = `"ชื่อที่แสดง": ("ชื่อวิทยาศาสตร์", "ชื่อสามัญ")`

- BioCLIP เรียนรู้จาก **ชื่อวิทยาศาสตร์** เป็นหลัก ถ้าเพิ่มชนิดใหม่ ให้ใส่ชื่อวิทยาศาสตร์ด้วยเสมอ
- **ลบชนิดที่ไม่มีทางเจอออก** ยิ่งตัวเลือกน้อย ยิ่งทายถูก (เช่น ถ้างานนี้มีแค่ดอกไม้จัดช่อ ไม่ต้องมี higanbana)

In [ ]:
import os, glob
from PIL import Image

# ---------- ด่าน "เป็นดอกไม้ไหม" (ใช้ BioCLIP 2 ไม่ต้องโหลด CLIP เพิ่ม) ----------
flower_prompts = [
    "a photo of a flower",
    "a close-up photo of a blooming flower",
    "a photo of a bouquet of flowers",
    "a photo of flowers in a garden",
]
other_prompts = [
    "a cartoon drawing", "a 2D game illustration", "a video game character", "an anime illustration",
    "a photo of an animal", "a photo of a person", "a photo of a face", "a photo of a hand",
    "a photo of food", "a photo of a building",   # ไม่มี "fruit": BioCLIP ดึงลีลาวดี/ทานตะวันจริงไปเป็นผลไม้
    "a photo of a car",
    "a photo of a landscape", "a photo of green leaves without flowers", "a photo of a tree",
    "a photo of an everyday object", "a screenshot", "a photo of text or a document",
    "a photo of a ribbon bow", "a photo of gift wrapping paper",   # YOLO (เทรนจากช่อดอกไม้) ชอบจับริบบิ้น/กระดาษห่อ
]
gate_prompts = flower_prompts + other_prompts
n_flower = len(flower_prompts)
gate_text = encode_text(bio_model, bio_tok, gate_prompts)
gate_scale = bio_model.logit_scale.exp().item()

# ---------- ชนิดดอกไม้: "ชื่อที่แสดง": ("ชื่อวิทยาศาสตร์", "ชื่อสามัญ") ----------
flowers = {
    # --- ดอกไม้จัดช่อยอดนิยม ---
    "rose": ("Rosa", "rose"),
    "tulip": ("Tulipa gesneriana", "tulip"),
    "lily": ("Lilium", "lily"),
    "orchid": ("Orchidaceae", "orchid"),
    "peony": ("Paeonia lactiflora", "peony"),
    "iris": ("Iris", "iris"),
    "chrysanthemum": ("Chrysanthemum morifolium", "chrysanthemum"),
    "lisianthus": ("Eustoma grandiflorum", "lisianthus"),
    "sunflower": ("Helianthus annuus", "sunflower"),
    "carnation": ("Dianthus caryophyllus", "carnation"),
    "baby's breath": ("Gypsophila paniculata", "baby's breath"),
    "gerbera": ("Gerbera jamesonii", "gerbera daisy"),
    "hydrangea": ("Hydrangea macrophylla", "hydrangea"),
    "daisy": ("Leucanthemum", "daisy"),
    "dahlia": ("Dahlia", "dahlia"),
    "ranunculus": ("Ranunculus asiaticus", "Persian buttercup"),
    "anemone": ("Anemone coronaria", "poppy anemone"),
    "freesia": ("Freesia", "freesia"),
    "gladiolus": ("Gladiolus", "gladiolus"),
    "alstroemeria": ("Alstroemeria", "Peruvian lily"),
    "calla lily": ("Zantedeschia", "calla lily"),
    "daffodil": ("Narcissus", "daffodil"),
    "hyacinth": ("Hyacinthus orientalis", "hyacinth"),
    "poppy": ("Papaver", "poppy"),
    "sweet pea": ("Lathyrus odoratus", "sweet pea"),
    "snapdragon": ("Antirrhinum majus", "snapdragon"),
    "delphinium": ("Delphinium", "larkspur"),
    "lilac": ("Syringa vulgaris", "lilac"),
    "lavender": ("Lavandula angustifolia", "lavender"),
    "stock": ("Matthiola incana", "stock"),
    "statice": ("Limonium sinuatum", "statice"),
    "wax flower": ("Chamelaucium uncinatum", "Geraldton wax flower"),
    "anthurium": ("Anthurium andraeanum", "flamingo flower"),
    "protea": ("Protea cynaroides", "king protea"),
    "aster": ("Aster", "aster"),
    "zinnia": ("Zinnia elegans", "zinnia"),
    "cosmos": ("Cosmos bipinnatus", "garden cosmos"),
    "lily of the valley": ("Convallaria majalis", "lily of the valley"),
    "forget-me-not": ("Myosotis", "forget-me-not"),
    # --- ดอกไม้ที่เจอบ่อยในไทย/เอเชีย ---
    "plumeria": ("Plumeria rubra", "frangipani"),          # ลีลาวดี
    "lotus": ("Nelumbo nucifera", "sacred lotus"),         # บัว
    "jasmine": ("Jasminum sambac", "Arabian jasmine"),     # มะลิ
    "marigold": ("Tagetes erecta", "African marigold"),    # ดาวเรือง
    "hibiscus": ("Hibiscus rosa-sinensis", "hibiscus"),    # ชบา
    "bougainvillea": ("Bougainvillea", "bougainvillea"),   # เฟื่องฟ้า
    "ixora": ("Ixora coccinea", "jungle geranium"),        # เข็ม
    "gardenia": ("Gardenia jasminoides", "gardenia"),
    "camellia": ("Camellia japonica", "camellia"),
    "magnolia": ("Magnolia", "magnolia"),
    "cherry blossom": ("Prunus serrulata", "Japanese cherry"),
    "higanbana": ("Lycoris radiata", "red spider lily"),
    # --- ดอกเติมช่อ / ช่อผสม ---
    "goldenrod": ("Solidago", "goldenrod"),
    "scabiosa": ("Scabiosa atropurpurea", "pincushion flower"),
    "clematis": ("Clematis", "clematis"),
    "craspedia": ("Craspedia globosa", "billy buttons"),
}

# prompt แบบที่ BioCLIP ใช้ตอนเทรน: ชื่อวิทยาศาสตร์ + ชื่อสามัญ
templates = [
    "a photo of {sci}.",
    "a photo of {sci} with common name {common}.",
    "a photo of {common} flowers.",
    "a close-up photo of a {common} flower.",
]
IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG", "*.WEBP")


def _text_emb(name):
    sci, common = flowers[name]
    f = encode_text(bio_model, bio_tok, [t.format(sci=sci, common=common) for t in templates]).mean(dim=0)
    return f / f.norm()


def _ref_paths(name):
    paths = []
    for ext in IMG_EXTS:
        paths += glob.glob(os.path.join(REF_DIR, name, ext))
    return sorted(set(paths))


def _ref_emb(paths):
    fs = []
    for p in paths:
        try:
            fs.append(encode_image(bio_model, bio_preprocess, Image.open(p).convert("RGB")))
        except Exception:
            print("⚠️ อ่านรูปตัวอย่างไม่ได้ ข้าม:", p)
    if not fs:
        return None
    f = torch.stack(fs).mean(dim=0)
    return f / f.norm()


def build_class_embs():
    """สร้าง embedding ของแต่ละชนิด จากข้อความ (+ รูปตัวอย่าง ถ้ามีโฟลเดอร์ refs/)"""
    global names, class_embs
    refs = {n: _ref_paths(n) for n in flowers}
    with_refs = [n for n in flowers if refs[n]]

    if os.path.isdir(REF_DIR):
        extra = [d for d in sorted(os.listdir(REF_DIR))
                 if os.path.isdir(os.path.join(REF_DIR, d)) and d not in flowers]
        if extra:
            print("⚠️ ชื่อโฟลเดอร์ไม่ตรงกับชื่อใน flowers (แก้ชื่อโฟลเดอร์ หรือเพิ่มชนิดใน flowers):", extra)

    if with_refs:
        # มีรูปตัวอย่าง → ทายเฉพาะชนิดที่มีรูปตัวอย่าง
        # (ถ้าปนกับชนิดที่มีแต่ข้อความ ชนิดที่มีรูปจะได้คะแนนสูงกว่าเสมอ ผลจะเอียง)
        names = with_refs
        embs = []
        for n in names:
            t, r = _text_emb(n), _ref_emb(refs[n])
            e = t if r is None else (1 - REF_WEIGHT) * t + REF_WEIGHT * r
            embs.append(e / e.norm())
        print(f"ใช้ข้อความ + รูปตัวอย่าง: {len(names)} ชนิด → "
              + ", ".join(f"{n} ({len(refs[n])} รูป)" for n in names))
    else:
        names = list(flowers)
        embs = [_text_emb(n) for n in names]
        print(f"ใช้ข้อความอย่างเดียว: {len(names)} ชนิด (ยังไม่มีรูปตัวอย่างใน {REF_DIR}/)")
    class_embs = torch.stack(embs)


build_class_embs()
bio_scale = bio_model.logit_scale.exp().item()
print("prompt ด่านกรอง:", len(gate_prompts))

## 5.1) (ไม่บังคับ แต่แม่นขึ้นมากที่สุด) อัปโหลดรูปตัวอย่าง
ทำไฟล์ `.zip` ที่ข้างในเป็นโฟลเดอร์ตามชื่อชนิด (ต้องตรงกับ key ใน `flowers`) เช่น

```
refs.zip
 └─ refs/
     ├─ tulip/      ← รูปทิวลิป 10–30 รูป (หลายสี หลายมุม ทั้งดอกตูมและบาน)
     ├─ rose/
     └─ calla lily/
```
- ใช้รูปที่ **เห็นดอกเดี่ยวชัดๆ** (crop มาแล้วยิ่งดี) และ **อย่าใช้รูปเดียวกับที่จะทดสอบ**
- เมื่อมีรูปตัวอย่าง ระบบจะทาย **เฉพาะชนิดที่มีโฟลเดอร์** เท่านั้น
- รันเซลล์นี้ซ้ำได้เพื่อเพิ่มรูป ระบบจะสร้าง embedding ใหม่ให้อัตโนมัติ

In [ ]:
import io, zipfile
try:
    from google.colab import files
except ImportError:   # VS Code / เครื่องตัวเอง: วางรูปใน refs/<ชนิด>/ ได้เลย ไม่ต้อง zip
    files = None


def upload_refs():
    if files is None:
        print(f"(VS Code) อ่านรูปตัวอย่างจากโฟลเดอร์ {REF_DIR}/<ชนิด>/ โดยตรง")
        build_class_embs()
        return
    up = files.upload()
    added = 0
    for fname, data in up.items():
        if not fname.lower().endswith(".zip"):
            print("⚠️ ข้าม (ไม่ใช่ .zip):", fname)
            continue
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            members = [m for m in z.namelist()
                       if not m.endswith("/") and not m.startswith("__MACOSX")
                       and not os.path.basename(m).startswith(".")]
            tops = {m.split("/")[0] for m in members}
            strip = len(tops) == 1 and next(iter(tops)) not in flowers   # zip มีโฟลเดอร์ครอบอีกชั้น เช่น refs/
            for m in members:
                parts = m.split("/")[1:] if strip else m.split("/")
                if len(parts) != 2 or ".." in parts:   # ต้องเป็น <ชนิด>/<รูป>
                    continue
                os.makedirs(os.path.join(REF_DIR, parts[0]), exist_ok=True)
                with open(os.path.join(REF_DIR, *parts), "wb") as f:
                    f.write(z.read(m))
                added += 1
    print(f"✅ เพิ่มรูปตัวอย่าง {added} รูป")
    build_class_embs()


upload_refs()

## 6) วิเคราะห์รูปเดียว: `analyze_image`

In [ ]:
def flower_prob(img, emb=None):
    """P(เป็นดอกไม้) จาก BioCLIP + สิ่งที่ใกล้เคียงที่สุดในกลุ่ม 'ไม่ใช่ดอกไม้'
    emb = embedding ของ img ที่คำนวณไว้แล้ว (ไม่ต้องเข้าโมเดลซ้ำ)"""
    g = encode_image(bio_model, bio_preprocess, img) if emb is None else emb
    p = (gate_scale * (g @ gate_text.T)).softmax(dim=-1)
    j = p[n_flower:].argmax().item()
    return p[:n_flower].sum().item(), (other_prompts[j], p[n_flower + j].item())


def species_logits(img, emb=None):
    b = encode_image(bio_model, bio_preprocess, img) if emb is None else emb
    return bio_scale * (class_embs @ b)


def classify_species(img, context=None, emb=None):
    """คืน (ชื่อชนิดหรือ UNKNOWN, ความมั่นใจ, top3)
    context = logits ของทั้งรูป เอามาผสม เพื่อให้ดอกในรูปเดียวกันไม่กระจัดกระจายไปคนละชนิด"""
    logits = species_logits(img, emb)
    if context is not None and CONTEXT_WEIGHT > 0:
        logits = (1 - CONTEXT_WEIGHT) * logits + CONTEXT_WEIGHT * context
    probs = logits.softmax(dim=-1)
    order = probs.argsort(descending=True).tolist()
    top3 = [(names[i], round(probs[i].item(), 4)) for i in order[:3]]
    best = top3[0]
    return (best[0] if best[1] >= SPECIES_THR else UNKNOWN), best[1], top3


import colorsys
import numpy as np


def flower_color(img, box):
    """สีหลักของดอก → ((r, g, b), ชื่อสี)
    ค่าเฉลี่ยทั้งกล่องจะปนสีใบไม้/พื้นหลัง และเฉลี่ยสีต่างกันจนได้สีที่ไม่มีอยู่จริง จึง
    ใช้กลางกล่อง 70% → ตัดใบไม้และพิกเซลมืด → เลือกโทนสีที่มีมากที่สุด → ค่ากลางของครึ่งที่สว่างกว่า"""
    x0, y0, x1, y1 = box
    dx, dy = (x1 - x0) * 0.15, (y1 - y0) * 0.15
    crop = img.crop((int(x0 + dx), int(y0 + dy), int(x1 - dx), int(y1 - dy)))
    rgb = np.asarray(crop, dtype=np.float32).reshape(-1, 3)
    h, s, v = (np.asarray(crop.convert("HSV"), dtype=np.float32).reshape(-1, 3) / 255).T   # PIL: hue 0–255 = 0–360°
    keep = (v > 0.2) & ~((h > 0.19) & (h < 0.56) & (s > 0.15))   # ไม่เอาใบไม้ (เขียว–เขียวอมฟ้า 70°–200°) และพิกเซลมืด
    if keep.sum() < 0.05 * len(rgb):                            # เหลือน้อยเกินไป (ดอกสีเขียว/มืด) → ใช้ทั้งหมด
        keep[:] = True
    pale = keep & (s < 0.15)
    if pale.sum() >= 0.5 * keep.sum():                          # ส่วนใหญ่ไม่มีสี → ดอกขาว/เทา
        pick = pale
    else:
        bins = (h * 12).astype(int) % 12                        # 12 โทน โทนละ 30°
        sat = keep & (s >= 0.15)
        pick = sat & (bins == np.bincount(bins[sat], minlength=12).argmax())
    pick &= v >= np.median(v[pick])                             # ครึ่งที่สว่างกว่า: คนดูสีจากกลีบที่โดนแสง ไม่ใช่เงา
    r, g, b = (int(x) for x in np.median(rgb[pick], axis=0))
    return (r, g, b), color_name(r, g, b)


def color_name(r, g, b):
    h, s, v = colorsys.rgb_to_hsv(r / 255, g / 255, b / 255)
    h *= 360
    if v < 0.2:
        return "black"
    if s < 0.15:
        return "white" if v > 0.7 else "grey"
    if h < 15 or h >= 345:
        return "pink" if s < 0.55 and v > 0.7 else "red"
    if h < 38:
        return "orange"
    if h < 70:
        return "yellow"
    if h < 170:
        return "green"
    if h < 255:
        return "blue"
    if h < 290 or v < 0.75:                                     # ม่วงอมชมพูเข้ม (ทิวลิปม่วง ~0.72) = purple, สว่าง (ชมพู ≥ 0.9) = pink
        return "purple"
    return "pink"


def merge_clusters(img, items, groups, ctx):
    """ดอกช่อ: กล่องที่ครอบหลายกล่องและทายได้เป็นชนิดใน CLUSTER_SPECIES → รวมดอกย่อยข้างในเป็น 1 ดอก
    ต้องมีดอกย่อยข้างใน ≥ ครึ่งเป็นชนิดเดียวกัน กันช่อผสมทั้งช่อถูกรวมเป็นดอกเดียว"""
    for box, score in sorted(groups, key=lambda g: (g[0][2] - g[0][0]) * (g[0][3] - g[0][1])):   # ช่อเล็กก่อน
        inner = [it for it in items if _covered(it["box"], box) >= NEST_COVER]
        if len(inner) < 2 or sum(1 for it in inner if it.get("cluster")) >= 2:   # ครอบหลายช่อ = ไม่ใช่ช่อเดียว
            continue
        label, conf, top3 = classify_species(crop_with_padding(img, box), ctx)
        if label not in CLUSTER_SPECIES or sum(it["label"] == label for it in inner) * 2 < len(inner):
            continue
        # ดอกย่อยชนิดเดียวกันที่ล้นขอบกล่องช่อออกไป (กล่องช่อมักสั้นกว่าช่อจริง) ก็เป็นช่อเดียวกัน
        inner += [it for it in items if it not in inner and it["label"] == label and _covered(it["box"], box) >= 0.25]
        items = [it for it in items if it not in inner]
        items.append({"box": [round(v) for v in box], "label": label, "conf": conf, "top3": top3,
                      "det_score": round(score, 3), "cluster": True})
        items[-1]["color"], items[-1]["color_name"] = flower_color(img, box)
    return items


def analyze_image(img):
    emb = encode_image(bio_model, bio_preprocess, img)
    p_img, looks_like = flower_prob(img, emb)
    if p_img < IMAGE_FLOWER_THR:
        return {"is_flower": False, "p_flower": p_img, "looks_like": looks_like,
                "items": [], "dropped": 0, "fallback": False, "reason": "not_flower"}

    ctx = species_logits(img, emb) if CONTEXT_WEIGHT > 0 else None   # บริบทของทั้งรูป
    if ctx is not None and ctx.softmax(dim=-1).max() < CONTEXT_MIN:       # ช่อผสมหลายชนิด → ไม่ใช้บริบท
        ctx = None
    items, dropped = [], 0
    boxes, groups = detect_flowers(img)
    for box, det_score in boxes:
        crop = crop_with_padding(img, box)
        e = encode_image(bio_model, bio_preprocess, crop)
        p_crop, _ = flower_prob(crop, e)
        if p_crop < CROP_FLOWER_THR:        # ไม่ใช่ดอกไม้จริง (ใบไม้ มือ ฯลฯ) → ตัดทิ้ง
            dropped += 1
            continue
        label, conf, top3 = classify_species(crop, ctx, e)
        rgb, cname = flower_color(img, box)
        items.append({"box": [round(v) for v in box], "label": label, "conf": conf,
                      "top3": top3, "det_score": round(det_score, 3), "color": rgb, "color_name": cname})
    if COUNT_CLUSTER_AS_ONE:
        items = merge_clusters(img, items, groups, ctx)

    fallback = False
    if not items:                           # detector ไม่เจอกล่อง → ประเมินชนิดจากทั้งรูป
        label, conf, top3 = classify_species(img, emb=emb)
        if label != UNKNOWN:
            W, H = img.size
            rgb, cname = flower_color(img, (0, 0, W, H))
            items.append({"box": [0, 0, W, H], "label": label, "conf": conf, "top3": top3, "det_score": None,
                          "color": rgb, "color_name": cname})
            fallback = True
        else:
            return {"is_flower": False, "p_flower": p_img, "looks_like": (top3[0][0], top3[0][1]),
                    "items": [], "dropped": dropped, "fallback": False, "reason": "not_flower"}

    return {"is_flower": True, "p_flower": p_img, "looks_like": None, "items": items,
            "dropped": dropped, "fallback": fallback}

## 7) ที่เก็บรูป: เพิ่มรูปได้หลายรอบก่อนกดตรวจ
**รันเซลล์นี้ซ้ำเพื่อเพิ่มรูปทีละชุด** (เลือกหลายไฟล์พร้อมกันได้) รูปจะสะสมไว้ในรายการ ยังไม่ตรวจจนกว่าจะรันเซลล์ที่ 8
ถ้าอยากล้างรายการแล้วเริ่มใหม่ ให้รัน `clear_images()`

In [ ]:
import io, os, glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
try:
    from google.colab import files
except ImportError:   # VS Code / เครื่องตัวเอง: อ่านรูปจากโฟลเดอร์แทนการอัปโหลด
    files = None

IMAGE_DIR = "../test_images"   # (VS Code) โฟลเดอร์ของรูปที่จะตรวจ
LOCAL_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".avif", ".bmp")

image_bank = []   # list ของ (ชื่อไฟล์, รูป PIL)


def clear_images():
    image_bank.clear()
    print("ล้างรายการรูปแล้ว")


def show_bank(cols=6):
    n = len(image_bank)
    if n == 0:
        print("ยังไม่มีรูป")
        return

    # คำนวณจำนวนแถวและคอลัมน์ให้ยืดหยุ่น
    actual_cols = min(n, cols)
    rows = (n + actual_cols - 1) // actual_cols if actual_cols > 0 else 1

    fig, axes = plt.subplots(rows, actual_cols, figsize=(2.2 * actual_cols, 2.2 * rows))

    # จัดการแกนแสดงผล (axes) ให้เป็น Flat list เสมอไม่ว่าจะอัปโหลดกี่รูป
    if n == 1:
        axes = [axes]
    elif hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for ax in axes:
        ax.axis("off")

    for ax, (name, im) in zip(axes, image_bank):
        ax.imshow(im)
        ax.set_title(name[:16], fontsize=7)
    plt.tight_layout()
    plt.show()


def add_images(folder=IMAGE_DIR):
    existing = {n for n, _ in image_bank}
    if files is not None:
        up = files.upload()
    else:   # รันซ้ำได้ จะเพิ่มเฉพาะไฟล์ใหม่ในโฟลเดอร์
        up = {os.path.basename(f): Path(f).read_bytes()
              for f in sorted(glob.glob(os.path.join(folder, "*")))
              if f.lower().endswith(LOCAL_EXTS) and os.path.basename(f) not in existing}
    for fname, data in up.items():
        try:
            im = Image.open(io.BytesIO(data)).convert("RGB")
        except Exception:
            print("⚠️ อ่านไฟล์ไม่ได้ ข้าม:", fname)
            continue
        name, k = fname, 2
        while name in existing:                  # ชื่อซ้ำ → ต่อท้ายเลข
            name = f"{fname.rsplit('.', 1)[0]}_{k}"
            k += 1
        existing.add(name)
        image_bank.append((name, im))
    print(f"✅ ตอนนี้มีรูปในรายการ {len(image_bank)} รูป")
    show_bank()


add_images()

## 8) กดตรวจทั้งหมด + สรุปรวม

In [ ]:
from collections import Counter, defaultdict
from IPython.display import display, HTML
import matplotlib.patches as patches


def summarize(results):
    """results = list ของ (ชื่อรูป, ผล analyze_image) → สรุปรวมทุกรูป (ฟังก์ชันล้วน ไม่แตะโมเดล)"""
    total, in_images, colors = Counter(), defaultdict(set), defaultdict(Counter)
    n_flower_imgs, rejected = 0, []
    for name, r in results:
        if not r["is_flower"]:
            rejected.append(name)
            continue
        n_flower_imgs += 1
        for it in r["items"]:
            total[it["label"]] += 1
            in_images[it["label"]].add(name)
            colors[it["label"]][it["color_name"]] += 1
    return {"total": total, "in_images": {k: len(v) for k, v in in_images.items()}, "colors": colors,
            "n_images": len(results), "n_flower_images": n_flower_imgs, "rejected": rejected,
            "n_flowers": sum(total.values())}


def draw_result(name, img, r):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.imshow(img)
    for i, it in enumerate(r["items"], 1):
        x0, y0, x1, y1 = it["box"]
        color = "orange" if it["label"] == UNKNOWN else "lime"
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=2))
        ax.text(x0, max(y0 - 4, 8), f"{i}. {it['color_name']} {it['label']}{' (cluster)' if it.get('cluster') else ''} {it['conf']:.0%}", fontsize=8, color="black",
                bbox=dict(facecolor=color, alpha=0.85, pad=1))
    ax.set_title(name, fontsize=9)
    ax.axis("off")
    plt.show()


def run_all():
    if not image_bank:
        print("ยังไม่มีรูป — รันเซลล์ที่ 7 เพื่อเพิ่มรูปก่อน")
        return None
    results = []
    for k, (name, img) in enumerate(image_bank, 1):
        print("=" * 60)
        print(f"[{k}/{len(image_bank)}] กำลังตรวจ {name} ...", flush=True)
        r = analyze_image(img)
        results.append((name, r))
        if not r["is_flower"]:
            what, pct = r["looks_like"]
            display(HTML(
                "<div style='background:#fdecea;border:2px solid #d93025;color:#b3261e;"
                "padding:10px;border-radius:8px'>"
                f"⚠️ <b>{name}</b>: ไม่พบดอกไม้จริงในรูปนี้ กรุณาเพิ่มรูปถ่ายดอกไม้จริงเข้ามา<br>"
                f"<span style='font-size:12px'>โอกาสเป็นดอกไม้ {r['p_flower']:.1%} "
                f"(ใกล้เคียง: {what} {pct:.1%})</span></div>"))
            continue
        draw_result(name, img, r)
        c = Counter(it["label"] for it in r["items"])
        txt = ", ".join(f"{k} {v}" for k, v in c.most_common())
        note = "  (detector ไม่เจอกล่อง ใช้ทั้งรูปเป็น 1 ดอก)" if r["fallback"] else ""
        extra = f"  [ตัดกล่องที่ไม่ใช่ดอกไม้ {r['dropped']} กล่อง]" if r["dropped"] else ""
        print(f"{name}: พบ {len(r['items'])} ดอก → {txt}{note}{extra}")
        if SHOW_TOP3:
            for i, it in enumerate(r["items"], 1):
                alts = ", ".join(f"{n} {p:.0%}" for n, p in it["top3"])
                print(f"   {i}. [{it['color_name']} RGB{it['color']}] {alts}")

    s = summarize(results)
    rows = "".join(
        f"<tr><td>{k}</td><td style='text-align:center'>{v}</td>"
        f"<td style='text-align:center'>{s['in_images'][k]}</td>"
        f"<td>{', '.join(f'{c} {n}' for c, n in s['colors'][k].most_common())}</td></tr>"
        for k, v in s["total"].most_common())
    rej = ""
    if s["rejected"]:
        rej = ("<p style='color:#b3261e'>⚠️ รูปที่ไม่ใช่ดอกไม้ (" + str(len(s["rejected"])) + " รูป): "
               + ", ".join(s["rejected"]) + " — กรุณาเพิ่มรูปดอกไม้</p>")
    display(HTML(
        "<div style='background:#e6f4ea;border:2px solid #188038;color:#0d3b1f;"
        "padding:14px;border-radius:8px;margin-top:12px'>"
        f"<h3 style='margin:0 0 8px 0'>🌸 สรุปรวม: {s['n_flowers']} ดอก จาก {s['n_flower_images']}/{s['n_images']} รูป</h3>"
        "<table style='border-collapse:collapse;min-width:320px'>"
        "<tr><th style='text-align:left'>ชนิด</th><th>จำนวนดอก</th><th>อยู่ในกี่รูป</th><th style='text-align:left'>สี</th></tr>"
        f"{rows}</table>{rej}</div>"))
    return results


results = run_all()